# Run the study on Colab

**This notebook is the primary runner.** All 3DGS optimization happens here;
the Mac is for editing code, reading results, and building figures.

## Storage layout, and why

| what | where | why |
|---|---|---|
| dataset **archives** | Drive | downloaded once, survive every session |
| dataset **extracted** | `/content` | Drive FUSE is slow for thousands of small PNGs; local disk is not |
| **runs** (working) | `/content` | training writes constantly; FUSE would throttle it |
| **runs** (persisted) | Drive | synced after each stage so `--resume` survives a disconnect |

The Kingston SSD cannot be read from Colab at all — only Drive is reachable
from a Colab VM, so the datasets have to live there.

Run the cells in order. After a disconnect, re-run everything except **Fetch
data** (Drive already has the archives) — extraction from a local archive
takes seconds.

In [ ]:
#@title 1. What GPU did we get?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
#@title 2. Mount Drive and set paths
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/dl3dcv'  #@param {type:'string'}

# Persistent (Drive) -- survives session death
ARCHIVES   = f'{PROJECT}/datasets/blender'
RUNS_DRIVE = f'{PROJECT}/runs'
# Working (local disk) -- fast, rebuilt each session
os.environ['DATA_ROOT'] = '/content/data'
os.environ['RUNS_ROOT'] = '/content/runs'

for d in (ARCHIVES, RUNS_DRIVE, '/content/runs'):
    os.makedirs(d, exist_ok=True)
print('archives  ', ARCHIVES)
print('DATA_ROOT ', os.environ['DATA_ROOT'], '(local)')
print('RUNS_ROOT ', os.environ['RUNS_ROOT'], '(local, synced to Drive)')

In [ ]:
#@title 3. Clone and install
REPO = 'https://github.com/AMEER7525/depth-prior-tax.git'  #@param {type:'string'}
%cd /content
![ -d final_project ] || git clone $REPO final_project
%cd /content/final_project
!git pull --ff-only || true
!bash scripts/setup_gpu.sh

## Fetch data

Downloads happen **here, not on your laptop** — Colab pulls from HuggingFace
at datacenter speed, straight into Drive. Downloading locally and syncing up
was measured at roughly 100 KB/s, which would take hours for the same data.

Skip this cell on later sessions; the next cell re-extracts from Drive.

In [ ]:
#@title 4. Download archives to Drive + extract to local disk
SCENES = 'lego chair ship'  #@param {type:'string'}
!python scripts/fetch_data.py --scenes $SCENES \
    --archives "$ARCHIVES" --extract "$DATA_ROOT"

In [ ]:
#@title 5. Sanity-check the loaders against real files
# src/data.py was written from the spec, not from the files. This is the
# first time it meets actual data -- check before spending GPU hours.
import sys; sys.path.insert(0, '/content/final_project')
import numpy as np
from src.data import load_blender_scene

sc = load_blender_scene(scene=SCENES.split()[0], split='train')
v = sc.views[0]
print(f'{len(sc)} views | image {v.image.shape} {v.image.dtype} '
      f'range [{v.image.min():.2f}, {v.image.max():.2f}]')
print('K =\n', np.round(v.K, 2))
print('c2w =\n', np.round(v.c2w, 3))
print('camera distance from origin:', np.round(np.linalg.norm(v.c2w[:3, 3]), 3),
      '(NeRF-Synthetic cameras sit ~4.0 from the object)')
print('has GT depth:', sc.has_depth, '<- False is expected; Axis C needs it rendered')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(11, 4))
for a, i in zip(ax, range(3)):
    a.imshow(sc.views[i].image); a.set_title(sc.views[i].name); a.axis('off')
plt.tight_layout()

## Run

Stages are sequential: each fixes the previous one's winner. Edit
`configs/sweep.yaml` between stages. `--resume` skips finished runs, so
re-running this cell after a disconnect picks up where it stopped.

In [ ]:
#@title 6. Restore finished runs from Drive (do this before running)
!mkdir -p "$RUNS_ROOT" && rsync -a "$RUNS_DRIVE/" "$RUNS_ROOT/" 2>/dev/null || true
!find "$RUNS_ROOT" -name metrics.json 2>/dev/null | wc -l | xargs echo 'finished runs restored:'

In [ ]:
#@title 7. Launch a stage
STAGE = 'stage1'  #@param ['stage1','stage2','stage3_noise','stage3_affine','stage4_real']
!python scripts/run_sweep.py --sweep configs/sweep.yaml --stage $STAGE \
    --resume --explain-pruning

In [ ]:
#@title 8. Sync results back to Drive (ALWAYS run before closing)
# Colab reclaims /content without warning. Anything not synced is gone.
!rsync -a "$RUNS_ROOT/" "$RUNS_DRIVE/"
!find "$RUNS_DRIVE" -name metrics.json | wc -l | xargs echo 'runs safe on Drive:'
!find "$RUNS_ROOT" -name FAILED | sed 's|.*/||' | head -20